# 03 — Review polarity (invert or keep)

Build a self-contained HTML sheet of **candidates** (Otsu: majority of pixels
are the dark class). Each row is the baked recto beside a **grayscale**
invert — that is the polarity we would commit, without the blue RGB cast.

Click a row (or press Space) to mark **invert**. Unmarked means keep as-is.
Marks are stored in the browser (`localStorage`). Export a CSV when done;
nothing is written to `pages-recto/` until a later apply step.

A handful of clearly normal pages are appended as controls so a false
invert is obvious.

In [ ]:
from pathlib import Path
import base64, csv, io, json, random

import numpy as np
from PIL import Image, ImageOps

Image.MAX_IMAGE_PIXELS = None

ROOT = Path("..").resolve()
RECTO = ROOT / "images" / "pages-recto"
OUT_HTML = ROOT / "outputs" / "polarity_review.html"
OUT_LIST = ROOT / "data" / "polarity_candidates.csv"

DARK_FRAC_MIN = 0.50
N_CONTROLS = 8
THUMB = 320
SEED = 0


In [ ]:
def otsu_dark_frac(gray_u8):
    hist = np.bincount(gray_u8.ravel(), minlength=256).astype(np.float64)
    total = hist.sum()
    levels = np.arange(256, dtype=np.float64)
    w0 = np.cumsum(hist)
    w1 = total - w0
    m0 = np.cumsum(hist * levels)
    sum_all = float((hist * levels).sum())
    ok = (w0 > 0) & (w1 > 0)
    between = np.zeros(256)
    between[ok] = ((sum_all * w0[ok] - m0[ok] * total) ** 2
                   / (w0[ok] * w1[ok] * total * total))
    t = int(np.argmax(between))
    return t, float((gray_u8 < t).mean())


def score(path, long=384):
    im = Image.open(path).convert("L")
    w, h = im.size
    s = long / max(w, h)
    small = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)
    a = np.asarray(small, np.uint8)
    t, dark = otsu_dark_frac(a)
    return {
        "name": path.name,
        "mean": float(a.mean()),
        "median": float(np.median(a)),
        "otsu": t,
        "dark_frac": dark,
    }


def jpeg_b64(im, long=THUMB, quality=78):
    t = im.copy()
    t.thumbnail((long, long))
    buf = io.BytesIO()
    t.save(buf, "JPEG", quality=quality)
    return base64.b64encode(buf.getvalue()).decode()


files = sorted(RECTO.glob("*.png"))
assert files, f"no PNGs in {RECTO}"
rows = [score(p) for p in files]
for r, p in zip(rows, files):
    r["path"] = p

cands = [r for r in rows if r["dark_frac"] >= DARK_FRAC_MIN]
cands.sort(key=lambda r: -r["dark_frac"])
normals = [r for r in rows if r["dark_frac"] < 0.30]
random.seed(SEED)
controls = random.sample(normals, min(N_CONTROLS, len(normals)))

print(f"rectos: {len(rows)}")
print(f"candidates (dark_frac >= {DARK_FRAC_MIN}): {len(cands)}")
print(f"controls: {len(controls)}")


In [ ]:
def pack(r, kind):
    rgb = Image.open(r["path"]).convert("RGB")
    ginv = ImageOps.invert(rgb.convert("L"))
    return {
        "name": r["name"],
        "kind": kind,
        "dark_frac": round(r["dark_frac"], 3),
        "mean": round(r["mean"], 1),
        "orig": jpeg_b64(rgb),
        "inv": jpeg_b64(ginv.convert("RGB")),
    }

items = [pack(r, "candidate") for r in cands] + [pack(r, "control") for r in controls]

OUT_LIST.parent.mkdir(parents=True, exist_ok=True)
with OUT_LIST.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=["filename", "kind", "dark_frac", "mean", "otsu"])
    w.writeheader()
    for r in cands:
        w.writerow({"filename": r["name"], "kind": "candidate",
                    "dark_frac": f"{r['dark_frac']:.4f}", "mean": f"{r['mean']:.1f}",
                    "otsu": r["otsu"]})
    for r in controls:
        w.writerow({"filename": r["name"], "kind": "control",
                    "dark_frac": f"{r['dark_frac']:.4f}", "mean": f"{r['mean']:.1f}",
                    "otsu": r["otsu"]})

payload = json.dumps(items, separators=(",", ":"))
html = r"""<!doctype html>
<meta charset=utf-8>
<title>Polarity review</title>
<style>
  :root { --bg:#111; --fg:#eee; --mut:#9ab; --acc:#3ddc84; --warn:#f0883e; }
  body { font-family: system-ui, sans-serif; margin: 0; background: var(--bg); color: var(--fg); }
  header { position: sticky; top: 0; z-index: 2; background: #1a1a1a; border-bottom: 1px solid #333;
           padding: 12px 20px; display: flex; gap: 16px; align-items: center; flex-wrap: wrap; }
  h1 { font-size: 16px; margin: 0; font-weight: 600; }
  .sub { color: var(--mut); font-size: 13px; }
  button { background: #333; color: var(--fg); border: 1px solid #555; border-radius: 6px;
           padding: 6px 12px; cursor: pointer; font: inherit; }
  button:hover { background: #444; }
  #sheet { padding: 16px 20px 64px; }
  .card { display: grid; grid-template-columns: 160px 1fr 1fr; gap: 12px; align-items: start;
          padding: 12px 0; border-bottom: 1px solid #2a2a2a; cursor: pointer; }
  .card.on { background: #14281c; outline: 1px solid var(--acc); }
  .card.control { opacity: 0.85; }
  .card img { width: 100%; max-width: 320px; display: block; border-radius: 4px; background: #222; }
  .meta { font: 12px ui-monospace, monospace; color: var(--mut); }
  .meta b { color: var(--fg); font-size: 14px; }
  .tag { display: inline-block; margin-top: 8px; padding: 2px 8px; border-radius: 999px; font-size: 11px; }
  .tag.keep { background: #333; }
  .tag.inv { background: var(--acc); color: #111; font-weight: 600; }
  .tag.ctrl { background: #3a3a00; color: #ee8; }
  .lbl { font-size: 11px; color: var(--mut); margin-bottom: 4px; }
  .k { color: #666; font-size: 12px; }
</style>
<header>
  <div>
    <h1>Polarity review</h1>
    <div class="sub">Click a row or press <b>Space</b> to mark invert &nbsp;·&nbsp;
      <span class="k">j / k</span> move &nbsp;·&nbsp; unmarked = do not invert</div>
  </div>
  <div class="sub" id="stats"></div>
  <button type="button" id="export">Export CSV</button>
  <button type="button" id="clear">Clear marks</button>
</header>
<div id="sheet"></div>
<script>
const ITEMS = __PAYLOAD__;
const KEY = "sluis-polarity-v1";
let marks = {};
try { marks = JSON.parse(localStorage.getItem(KEY) || "{}"); } catch (e) { marks = {}; }
let focus = 0;

function save() { localStorage.setItem(KEY, JSON.stringify(marks)); stats(); }
function isOn(name) { return !!marks[name]; }
function stats() {
  const n = ITEMS.filter(it => it.kind === "candidate").length;
  const m = ITEMS.filter(it => it.kind === "candidate" && isOn(it.name)).length;
  document.getElementById("stats").textContent =
    m + " marked invert / " + n + " candidates  (" + ITEMS.filter(it => it.kind==="control").length + " controls)";
}
function render() {
  const sheet = document.getElementById("sheet");
  sheet.innerHTML = ITEMS.map((it, i) => {
    const on = isOn(it.name);
    const tag = it.kind === "control" ? "<span class='tag ctrl'>control (should stay as-is)</span>"
              : on ? "<span class='tag inv'>INVERT</span>" : "<span class='tag keep'>do not invert</span>";
    return `<div class="card ${on?"on":""} ${it.kind}" data-i="${i}" id="c${i}">
      <div class="meta"><b>${it.name}</b><br>dark ${it.dark_frac} · mean ${it.mean}<br>${tag}</div>
      <div><div class="lbl">current (no invert)</div><img src="data:image/jpeg;base64,${it.orig}"></div>
      <div><div class="lbl">grayscale invert (proposed)</div><img src="data:image/jpeg;base64,${it.inv}"></div>
    </div>`;
  }).join("");
  sheet.querySelectorAll(".card").forEach(el => {
    el.addEventListener("click", () => toggle(+el.dataset.i));
  });
  highlight();
  stats();
}
function toggle(i) {
  const name = ITEMS[i].name;
  if (marks[name]) delete marks[name]; else marks[name] = 1;
  focus = i;
  save();
  const el = document.getElementById("c"+i);
  el.classList.toggle("on", isOn(name));
  const tag = el.querySelector(".tag");
  if (ITEMS[i].kind === "control") return;
  tag.className = "tag " + (isOn(name) ? "inv" : "keep");
  tag.textContent = isOn(name) ? "INVERT" : "do not invert";
}
function highlight() {
  document.querySelectorAll(".card").forEach((el, i) => {
    el.style.boxShadow = i === focus ? "inset 3px 0 0 #6ae" : "";
  });
}
function goto(i) {
  focus = Math.max(0, Math.min(ITEMS.length - 1, i));
  highlight();
  document.getElementById("c"+focus).scrollIntoView({block: "nearest"});
}
document.getElementById("export").onclick = () => {
  const lines = ["filename,invert,kind,dark_frac"];
  ITEMS.forEach(it => lines.push([it.name, isOn(it.name) ? 1 : 0, it.kind, it.dark_frac].join(",")));
  const blob = new Blob([lines.join("\n")], {type: "text/csv"});
  const a = document.createElement("a");
  a.href = URL.createObjectURL(blob);
  a.download = "polarity_decisions.csv";
  a.click();
};
document.getElementById("clear").onclick = () => {
  if (!confirm("Clear all invert marks?")) return;
  marks = {}; save(); render();
};
document.addEventListener("keydown", e => {
  if (e.target.tagName === "INPUT") return;
  if (e.key === "j" || e.key === "ArrowDown") { e.preventDefault(); goto(focus + 1); }
  if (e.key === "k" || e.key === "ArrowUp") { e.preventDefault(); goto(focus - 1); }
  if (e.key === " " || e.key === "Enter") { e.preventDefault(); toggle(focus); }
});
render();
</script>
"""
OUT_HTML.parent.mkdir(parents=True, exist_ok=True)
OUT_HTML.write_text(html.replace("__PAYLOAD__", payload), encoding="utf-8")
print(f"candidates list → {OUT_LIST}")
print(f"review sheet    → {OUT_HTML}")
print(f"size {OUT_HTML.stat().st_size / 1e6:.1f} MB")
